In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/04_rag_retrieval.py

# Phase 14.6 — RAG Evaluation

This notebook evaluates the RAG pipeline.

Evaluation areas:

1. Retrieval availability
2. Retrieval relevance
3. Context quality
4. Answer generation
5. Citation availability
6. Unsupported-question handling
7. Overall RAG quality

The evaluation uses deterministic checks where possible.


In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/05_rag_answer_generation.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/04_rag_retrieval.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/05_rag_answer_generation.py

In [0]:
import json
import time

from pyspark.sql import functions as F

print("Imports successful.")

In [0]:
EMBEDDINGS_TABLE = "genai_copilot.gold.document_embeddings"

TOP_K = 3

print("Embeddings table:", EMBEDDINGS_TABLE)
print("TOP_K:", TOP_K)

In [0]:
evaluation_questions = [
    {
        "id": "RAG001",
        "question": "What is the discount policy?",
        "expected_type": "supported"
    },
    {
        "id": "RAG002",
        "question": "What are the rules for discounts?",
        "expected_type": "supported"
    },
    {
        "id": "RAG003",
        "question": "What is the employee vacation policy?",
        "expected_type": "unsupported"
    }
]

print("Evaluation questions:", len(evaluation_questions))

In [0]:
def retrieve_for_evaluation(question):
    """
    Generate a question embedding and retrieve
    the most relevant document chunks.
    """

    question_embedding = generate_question_embedding(
        question
    )

    results = []

    for row in document_rows:

        similarity = cosine_similarity(
            question_embedding,
            row["embedding"]
        )

        results.append(
            {
                "chunk_id": row["chunk_id"],
                "document_id": row["document_id"],
                "file_name": row["file_name"],
                "document_type": row["document_type"],
                "title": row["title"],
                "chunk_index": row["chunk_index"],
                "chunk_text": row["chunk_text"],
                "similarity": float(similarity)
            }
        )

    results = sorted(
        results,
        key=lambda x: x["similarity"],
        reverse=True
    )

    return results[:TOP_K]

In [0]:
def evaluate_retrieval(
    question,
    expected_type
):
    start_time = time.time()

    retrieved = retrieve_for_evaluation(
        question
    )

    retrieval_time_ms = round(
        (time.time() - start_time) * 1000,
        2
    )

    retrieved_count = len(retrieved)

    similarities = [
        item["similarity"]
        for item in retrieved
    ]

    average_similarity = (
        sum(similarities) / len(similarities)
        if similarities
        else 0.0
    )

    top_similarity = (
        similarities[0]
        if similarities
        else 0.0
    )

    retrieval_available = (
        retrieved_count > 0
    )

    return {
        "retrieved": retrieved,
        "retrieved_count": retrieved_count,
        "top_similarity": top_similarity,
        "average_similarity": average_similarity,
        "retrieval_time_ms": retrieval_time_ms,
        "retrieval_available": retrieval_available,
        "expected_type": expected_type
    }

In [0]:
evaluation_results = []

for test in evaluation_questions:

    retrieval = evaluate_retrieval(
        test["question"],
        test["expected_type"]
    )

    evaluation_results.append(
        {
            "id": test["id"],
            "question": test["question"],
            **retrieval
        }
    )

    print(
        f"{test['id']} | "
        f"top similarity: "
        f"{retrieval['top_similarity']:.4f} | "
        f"retrieved: "
        f"{retrieval['retrieved_count']}"
    )

In [0]:
retrieval_summary = [
    {
        "id": item["id"],
        "question": item["question"],
        "expected_type": item["expected_type"],
        "retrieved_count": item["retrieved_count"],
        "top_similarity": item["top_similarity"],
        "average_similarity": item["average_similarity"],
        "retrieval_time_ms": item["retrieval_time_ms"]
    }
    for item in evaluation_results
]

retrieval_summary_df = spark.createDataFrame(
    retrieval_summary
)

display(
    retrieval_summary_df
    .orderBy("id")
)

In [0]:
MIN_RETRIEVAL_SIMILARITY = 0.50

In [0]:
MIN_RETRIEVAL_SIMILARITY = 0.50

print(
    "Minimum retrieval similarity:",
    MIN_RETRIEVAL_SIMILARITY
)

In [0]:
for item in evaluation_results:

    if item["expected_type"] == "supported":

        passed = (
            item["retrieval_available"]
            and
            item["top_similarity"]
            >= MIN_RETRIEVAL_SIMILARITY
        )

    else:

        passed = item["retrieval_available"]

    item["retrieval_pass"] = passed

    print(
        f"{item['id']}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

In [0]:
def evaluate_answer(question, retrieval):
    """
    Generate an answer using retrieved context.
    """

    context = build_rag_context(
        retrieval["retrieved"]
    )

    answer_result = generate_rag_answer(
        question,
        context
    )

    citations = build_source_citations(
        retrieval["retrieved"]
    )

    answer = answer_result.get(
        "answer",
        ""
    )

    return {
        "answer": answer,
        "context": context,
        "citations": citations,
        "latency_ms": answer_result.get(
            "latency_ms",
            0
        )
    }

In [0]:
for item in evaluation_results:

    answer_result = evaluate_answer(
        item["question"],
        item
    )

    item["answer"] = answer_result["answer"]
    item["context"] = answer_result["context"]
    item["citations"] = answer_result["citations"]
    item["answer_latency_ms"] = answer_result["latency_ms"]

    item["answer_generated"] = bool(
        item["answer"].strip()
    )

    item["citations_available"] = (
        len(item["citations"]) > 0
    )

In [0]:
for item in evaluation_results:

    print("=" * 70)
    print(item["id"])
    print("Question:", item["question"])
    print()
    print("Answer:")
    print(item["answer"])
    print()
    print("Sources:")

    for source in item["citations"]:
        print(
            f"- {source['file_name']} | "
            f"{source['title']} | "
            f"{source['similarity']:.4f}"
        )

    print("=" * 70)

Unsupported question check

This is very important.

For RAG003, the system should not hallucinate a vacation policy.

In [0]:
unsupported_test = next(
    item
    for item in evaluation_results
    if item["expected_type"] == "unsupported"
)

unsupported_answer = (
    unsupported_test["answer"]
)

print("Unsupported question:")
print(
    unsupported_test["question"]
)

print("\nGenerated answer:")
print(unsupported_answer)

In [0]:
refusal_phrases = [
    "do not contain enough information",
    "not enough information",
    "cannot answer",
    "not available",
    "not provided",
    "no information",
    "unable to answer",
    "available documents"
]

unsupported_lower = (
    unsupported_answer.lower()
)

unsupported_guard_pass = any(
    phrase in unsupported_lower
    for phrase in refusal_phrases
)

unsupported_test["unsupported_guard_pass"] = (
    unsupported_guard_pass
)

print(
    "Unsupported-question guard:",
    "PASS"
    if unsupported_guard_pass
    else "FAIL"
)

In [0]:
for item in evaluation_results:

    citations = item["citations"]

    valid_citations = True

    for citation in citations:

        if not citation.get("file_name"):
            valid_citations = False

        if not citation.get("chunk_id"):
            valid_citations = False

    item["citation_validation_pass"] = (
        valid_citations
        and len(citations) > 0
    )

In [0]:
for item in evaluation_results:

    if item["expected_type"] == "supported":

        item["overall_pass"] = (
            item["retrieval_pass"]
            and item["answer_generated"]
            and item["citations_available"]
            and item["citation_validation_pass"]
        )

    else:

        item["overall_pass"] = (
            item["unsupported_guard_pass"]
        )

    print(
        item["id"],
        "→",
        "PASS"
        if item["overall_pass"]
        else "FAIL"
    )

In [0]:
total_tests = len(
    evaluation_results
)

passed_tests = sum(
    1
    for item in evaluation_results
    if item["overall_pass"]
)

failed_tests = (
    total_tests - passed_tests
)

overall_status = (
    "PASS"
    if failed_tests == 0
    else "FAIL"
)

print("=" * 70)
print("PHASE 14.6 — RAG EVALUATION")
print("=" * 70)

print("Total tests:", total_tests)
print("Passed tests:", passed_tests)
print("Failed tests:", failed_tests)
print("Overall status:", overall_status)

print("=" * 70)

In [0]:
evaluation_table = []

for item in evaluation_results:

    evaluation_table.append(
        {
            "id": item["id"],
            "question": item["question"],
            "expected_type": item["expected_type"],
            "retrieved_count": item["retrieved_count"],
            "top_similarity": item["top_similarity"],
            "average_similarity": item["average_similarity"],
            "retrieval_pass": item["retrieval_pass"],
            "answer_generated": item["answer_generated"],
            "citations_available": item["citations_available"],
            "citation_validation_pass": item[
                "citation_validation_pass"
            ],
            "unsupported_guard_pass": item.get(
                "unsupported_guard_pass",
                True
            ),
            "overall_pass": item["overall_pass"]
        }
    )

evaluation_df = spark.createDataFrame(
    evaluation_table
)

display(
    evaluation_df.orderBy("id")
)

In [0]:
print("=" * 70)
print("RAG QUALITY GATE")
print("=" * 70)

if overall_status == "PASS":

    print("✓ Retrieval quality checks passed")
    print("✓ Answer generation checks passed")
    print("✓ Citation checks passed")
    print("✓ Unsupported-question guard passed")
    print()
    print("✓ RAG QUALITY GATE: PASS")

else:

    print("✗ One or more RAG evaluation checks failed")
    print("✗ Review the evaluation table")
    print()
    print("✗ RAG QUALITY GATE: FAIL")

print("=" * 70)

overall copilot architecture is now:

                         USER QUESTION
                               │
                               ▼
                    QUESTION CLASSIFIER
                         /           \
                        /             \
                       ▼               ▼
                  SQL PATH          RAG PATH
                       │               │
                 SQL Generator    Embeddings
                       │               │
                 SQL Validator     Retrieval
                       │               │
                 SQL Executor      RAG Context
                       │               │
                       └───────┬───────┘
                               ▼
                       ANSWER GENERATION
                               │
                               ▼
                         VISUALIZATION
                               │
                               ▼
                         FINAL RESPONSE